In [ ]:
# 2026-02-03 10:21 Europe/Berlin
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()

# Suche nach einem Parent, der einen "src/helpers" Ordner hat
SRC_DIR = None
for p in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    cand = p / "src" / "helpers"
    if cand.exists():
        SRC_DIR = p / "src"
        break

if SRC_DIR is None:
    raise FileNotFoundError(
        f"Konnte kein 'src/helpers' finden ab cwd={NOTEBOOK_DIR}. "
        "Prüfe, ob du im richtigen Projektordner bist."
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("cwd:", NOTEBOOK_DIR)
print("SRC_DIR:", SRC_DIR)
print("helpers exists:", (SRC_DIR / "helpers").exists())

from helpers.soll_path import load_soll_haltestellen
from helpers.vehicle_positions import load_vp_for_umlauf


In [ ]:
# 2026-02-02 18:10 Europe/Berlin

from pathlib import Path
from datetime import timedelta

# ============================================================
# 0) PARAMETER (nur hier anfassen)
# ============================================================

TAG = "2026-01-08"
UMLAUF_ID = 1494
SOURCE = "muenster"

# OSRM
OSRM_BASE_URL = "http://localhost:5000"
OSRM_PROFILE = "driving"
OSRM_TIMEOUT_S = 20

# Zeitfenster für VP-Load um gesamten Umlauf
TIME_GATE_LOAD = timedelta(minutes=10)

# --- Arrivals / Interpolation (wie in deinem letzten Block) ---
TIME_GATE = timedelta(minutes=10)

LIKELIHOOD_MAX_DIST_M = 100.0
SIGMA_DIST_M = 25.0
ANCHOR_DIST_M = 20.0
LOCAL_TZ = "Europe/Berlin"

EDGE_STAGE_1_TIME = timedelta(minutes=3)
EDGE_STAGE_1_DIST = 100.0
EDGE_STAGE_2_TIME = timedelta(minutes=10)
EDGE_STAGE_2_DIST = 300.0

STOP_ANCHOR_DIST_M = 75.0
STOP_ANCHOR_MAX_DT = timedelta(minutes=6)

# Export
NOTEBOOK_DIR = Path.cwd()
EXPORT_DIR = NOTEBOOK_DIR / "export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("EXPORT_DIR:", EXPORT_DIR)


In [ ]:
# 2026-02-02 18:10 Europe/Berlin

import sys
import webbrowser

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import folium
from folium.plugins import Fullscreen, MousePosition, MeasureControl
from shapely.geometry import Point, LineString

# ============================================================
# 1) PATHS + IMPORT HELPERS
# ============================================================

SRC_DIR = NOTEBOOK_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from helpers.soll_path import load_soll_haltestellen
from helpers.vehicle_positions import load_vp_for_umlauf


In [ ]:
# 2026-02-02 18:10 Europe/Berlin

def ensure_gdf(df, crs="EPSG:4326") -> gpd.GeoDataFrame:
    if isinstance(df, gpd.GeoDataFrame):
        if df.geometry is None:
            if "geometry" in df.columns:
                df = df.set_geometry("geometry")
            elif "geom" in df.columns:
                df = df.set_geometry("geom")
            else:
                raise ValueError(f"GeoDataFrame ohne geometry/geom. Spalten: {list(df.columns)}")
        if df.crs is None:
            df = df.set_crs(crs)
        return df

    if "geometry" in df.columns:
        return gpd.GeoDataFrame(df, geometry="geometry", crs=crs)
    if "geom" in df.columns:
        return gpd.GeoDataFrame(df, geometry="geom", crs=crs)

    raise ValueError(f"Kein geometry/geom Feld vorhanden. Spalten: {list(df.columns)}")


def to_metric(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf is None or len(gdf) == 0:
        return gdf
    gdf = ensure_gdf(gdf)
    return gdf.to_crs(gdf.estimate_utm_crs())


def normalize_time_utc(series: pd.Series, assume_local_if_naive: bool) -> pd.Series:
    """
    Liefert tz-aware UTC.
    - SOLL ist oft tz-naive -> als LOCAL_TZ interpretieren -> UTC
    - VP ist i.d.R. UTC -> assume_local_if_naive=False
    """
    s = pd.to_datetime(series, errors="coerce")
    tz = getattr(s.dt, "tz", None)
    if tz is None:
        if assume_local_if_naive:
            return s.dt.tz_localize(LOCAL_TZ).dt.tz_convert("UTC")
        return s.dt.tz_localize("UTC")
    return s.dt.tz_convert("UTC")


In [ ]:
# 2026-02-02 18:10 Europe/Berlin

def osrm_segments_between(p1, p2):
    url = (
        f"{OSRM_BASE_URL}/route/v1/{OSRM_PROFILE}/"
        f"{p1[0]},{p1[1]};{p2[0]},{p2[1]}"
        "?steps=true&geometries=geojson&overview=false"
    )
    r = requests.get(url, timeout=OSRM_TIMEOUT_S)
    r.raise_for_status()
    data = r.json()

    segments = []
    for leg in data["routes"][0]["legs"]:
        for step in leg["steps"]:
            segments.append(
                {
                    "geometry": LineString(step["geometry"]["coordinates"]),
                    "length_m": float(step["distance"]),
                }
            )
    return segments


def build_soll_segments(soll_stops_wgs84: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    soll_stops_wgs84 = ensure_gdf(soll_stops_wgs84, crs="EPSG:4326")

    rows = []
    for frt_fid, grp in (
        soll_stops_wgs84.sort_values(["frt_fid", "stop_seq"]).groupby("frt_fid")
    ):
        geom_col = grp.geometry.name
        coords = [(float(p.x), float(p.y)) for p in grp[geom_col]]  # (lon, lat)
        if len(coords) < 2:
            continue

        cum_dist = 0.0
        edge_idx = 0

        for p1, p2 in zip(coords[:-1], coords[1:]):
            for seg in osrm_segments_between(p1, p2):
                rows.append(
                    {
                        "frt_fid": frt_fid,
                        "edge_idx": edge_idx,
                        "geometry": seg["geometry"],
                        "cum_start_m": cum_dist,
                        "cum_end_m": cum_dist + seg["length_m"],
                    }
                )
                cum_dist += seg["length_m"]
                edge_idx += 1

    return gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")


# --- SOLL laden ---
soll_stops = load_soll_haltestellen(TAG, UMLAUF_ID)
soll_stops = ensure_gdf(soll_stops, crs="EPSG:4326")
soll_stops = soll_stops.sort_values(["frt_fid", "stop_seq"]).reset_index(drop=True)

# Zeit normieren (UTC-aware)
soll_stops["ts_soll"] = normalize_time_utc(soll_stops["ts_soll"], assume_local_if_naive=True)

print("SOLL stops:", len(soll_stops), "| Fahrten:", soll_stops["frt_fid"].nunique())
print("ts span:", soll_stops["ts_soll"].min(), "→", soll_stops["ts_soll"].max())

# --- OSRM Segmente bauen ---
soll_segments = build_soll_segments(soll_stops.to_crs("EPSG:4326"))
soll_segments = ensure_gdf(soll_segments, crs="EPSG:4326")

print("SOLL segments:", len(soll_segments), "| frt_fid:", soll_segments["frt_fid"].nunique())


In [ ]:
# 2026-02-03 11:28 Europe/Berlin

from datetime import timedelta
import numpy as np
import pandas as pd
import geopandas as gpd

# ============================================================
# 0) PRO UMLAUF Override (hier eintragen)
# ============================================================

# Beispiel:
# MANUAL_VEHICLE_BY_UMLAUF = {
#     1849: "1234",   # <- UMLAUF_ID: vehicle_id
# }
MANUAL_VEHICLE_BY_UMLAUF = {}

def _cast_vehicle_id_like(vp_df: gpd.GeoDataFrame | None, vehicle_id):
    """Sorgt dafür, dass der Typ zu vp['vehicle_id'] passt (z.B. int vs str)."""
    if vp_df is None or "vehicle_id" not in vp_df.columns:
        return vehicle_id
    try:
        s = vp_df["vehicle_id"].dropna()
        if len(s) == 0:
            return vehicle_id
        sample = s.iloc[0]
        return sample.__class__(vehicle_id)
    except Exception:
        return vehicle_id


# ============================================================
# 1) VP laden (Fenster um Umlauf)
# ============================================================

# Erwartet: soll_stops hat ts_soll (oder du normalisierst vorher)
# TIME_GATE_LOAD, TAG, SOURCE, UMLAUF_ID existieren

ts_start = soll_stops["ts_soll"].min() - TIME_GATE_LOAD
ts_end   = soll_stops["ts_soll"].max() + TIME_GATE_LOAD

vp = load_vp_for_umlauf(
    tag=TAG,
    ts_start=ts_start,
    ts_end=ts_end,
    source=SOURCE,
    vehicle_id=None,          # absichtlich: alle, weil wir best vehicle suchen
)

vp = ensure_gdf(vp, crs="EPSG:4326")
vp["ts"] = normalize_time_utc(vp["ts"], assume_local_if_naive=False)

print("VP points:", len(vp), "| vehicles:", vp["vehicle_id"].nunique())
print("VP ts span:", vp["ts"].min(), "→", vp["ts"].max())


# ============================================================
# 2) metrisch + sindex für Anchor Matching
# ============================================================

metric_crs = soll_stops.estimate_utm_crs()
soll_m = soll_stops.to_crs(metric_crs)
vp_m   = vp.to_crs(metric_crs)
sidx   = vp_m.sindex


# ============================================================
# 3) Parameter: Anchor Kandidaten
# ============================================================

ANCHOR_TIME_BUF  = timedelta(minutes=8)  # wie gewünscht
ANCHOR_DIST_M    = 200.0
K_NEAREST        = 10

W_DT_S, W_DIST_M = 1.0, 1.0
DT_SCALE_S       = 60.0
DIST_SCALE_M     = 50.0

MAX_DT_S         = 10 * 60
MAX_DIST_M       = 200.0


# ============================================================
# 4) Helpers
# ============================================================

def build_anchors_from_soll(soll_m: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Erzeuge Start/End-Anchor pro frt_fid (erste und letzte Haltestelle).
    Erwartet Spalten: frt_fid, stop_seq, ts_soll, geometry
    """
    soll_m = soll_m.copy().sort_values(["frt_fid", "stop_seq"]).reset_index(drop=True)
    anchors = []
    gc = soll_m.geometry.name

    for frt, grp in soll_m.groupby("frt_fid"):
        g = grp.sort_values("stop_seq")
        if len(g) == 0:
            continue

        first = g.iloc[0]
        last  = g.iloc[-1]

        anchors.append({
            "frt_fid": frt,
            "anchor_type": "start",
            "stop_seq": int(first["stop_seq"]),
            "ts_soll": first["ts_soll"],
            "geometry": first[gc],
        })
        anchors.append({
            "frt_fid": frt,
            "anchor_type": "end",
            "stop_seq": int(last["stop_seq"]),
            "ts_soll": last["ts_soll"],
            "geometry": last[gc],
        })

    return gpd.GeoDataFrame(anchors, geometry="geometry", crs=soll_m.crs)


def anchor_candidates(anchor_row: pd.Series, vp_m: gpd.GeoDataFrame, sidx) -> pd.DataFrame:
    """
    Kandidatenpunkte in Raum + Zeit um den Anchor (eine Stop-Geometry + ts_soll).
    Gibt ein DataFrame mit vehicle_id, ts, dist_m, dt_s zurück (Top K_NEAREST).
    """
    pt = anchor_row["geometry"]
    ts = anchor_row["ts_soll"]

    if pt is None or pt.is_empty or pd.isna(ts):
        return pd.DataFrame()

    # spatial prefilter
    buf = pt.buffer(ANCHOR_DIST_M)
    idx = list(sidx.intersection(buf.bounds)) if sidx is not None else []
    if not idx:
        return pd.DataFrame()

    cand = vp_m.iloc[idx].copy()
    cand["dist_m"] = cand.geometry.distance(pt)
    cand = cand[cand["dist_m"] <= ANCHOR_DIST_M].copy()
    if cand.empty:
        return pd.DataFrame()

    # time filter
    t0 = ts - ANCHOR_TIME_BUF
    t1 = ts + ANCHOR_TIME_BUF
    cand = cand[(cand["ts"] >= t0) & (cand["ts"] <= t1)].copy()
    if cand.empty:
        return pd.DataFrame()

    cand["dt_s"] = (cand["ts"] - ts).dt.total_seconds().abs()

    # hard caps
    cand = cand[(cand["dt_s"] <= MAX_DT_S) & (cand["dist_m"] <= MAX_DIST_M)].copy()
    if cand.empty:
        return pd.DataFrame()

    # keep nearest points (prefer distance first, then time)
    cand = cand.sort_values(["dist_m", "dt_s"]).head(K_NEAREST).copy()
    cols = [c for c in ["vehicle_id", "ts", "dist_m", "dt_s"] if c in cand.columns]
    return cand[cols]


def pick_best_vehicle_for_anchor(cand: pd.DataFrame):
    """
    Erzeugt einen Score pro Kandidatenpunkt und wählt pro vehicle_id den besten Punkt.
    Rückgabe: (best_vehicle_id, best_per_vehicle_table)
    """
    if cand is None or len(cand) == 0:
        return None, pd.DataFrame()

    c = cand.copy()
    c["score_pt"] = (
        W_DT_S   * (c["dt_s"]   / DT_SCALE_S) +
        W_DIST_M * (c["dist_m"] / DIST_SCALE_M)
    )

    best_per_vehicle = (
        c.sort_values(["vehicle_id", "score_pt", "dt_s", "dist_m"])
         .groupby("vehicle_id", as_index=False)
         .head(1)
         .sort_values("score_pt")
         .reset_index(drop=True)
    )

    best_vehicle_final = best_per_vehicle.iloc[0]["vehicle_id"] if len(best_per_vehicle) else None
    return best_vehicle_final, best_per_vehicle


def match_vehicles_via_anchors(soll_m: gpd.GeoDataFrame, vp_m: gpd.GeoDataFrame, sidx):
    """
    Läuft alle Start/End-Anker ab, sammelt best_vehicle_final pro Anker,
    aggregiert dann global den besten vehicle_id-Kandidaten.
    """
    anchors = build_anchors_from_soll(soll_m)

    rows = []
    for _, a in anchors.iterrows():
        cand = anchor_candidates(a, vp_m, sidx)
        best_vid, best_tbl = pick_best_vehicle_for_anchor(cand)

        rows.append({
            "frt_fid": a["frt_fid"],
            "anchor_type": a["anchor_type"],
            "stop_seq": a["stop_seq"],
            "ts_soll": a["ts_soll"],
            "best_vehicle_id": best_vid,
            "n_cand_points": int(len(cand)) if cand is not None else 0,
            "n_cand_vehicles": int(cand["vehicle_id"].nunique()) if cand is not None and len(cand) else 0,
            "best_score": float(best_tbl.iloc[0]["score_pt"]) if len(best_tbl) else np.nan,
            "best_dt_s": float(best_tbl.iloc[0]["dt_s"]) if len(best_tbl) else np.nan,
            "best_dist_m": float(best_tbl.iloc[0]["dist_m"]) if len(best_tbl) else np.nan,
        })

    df_anchor = pd.DataFrame(rows)

    agg = (
        df_anchor.dropna(subset=["best_vehicle_id"])
        .groupby("best_vehicle_id")
        .agg(
            anchor_hits=("best_vehicle_id", "size"),
            frt_covered=("frt_fid", "nunique"),
            score_mean=("best_score", "mean"),
            dt_mean_s=("best_dt_s", "mean"),
            dist_mean_m=("best_dist_m", "mean"),
        )
        .reset_index()
        .sort_values(["anchor_hits", "frt_covered"], ascending=[False, False])
    )

    best_day_vehicle = agg.iloc[0]["best_vehicle_id"] if len(agg) else None
    return best_day_vehicle, df_anchor, agg


# ============================================================
# 5) RUN + PRO-UMLAUF Override
# ============================================================

best_vehicle_auto, df_anchor, agg_vehicle = match_vehicles_via_anchors(soll_m, vp_m, sidx)

best_vehicle_final = best_vehicle_auto
if UMLAUF_ID in MANUAL_VEHICLE_BY_UMLAUF:
    best_vehicle_final = _cast_vehicle_id_like(vp, MANUAL_VEHICLE_BY_UMLAUF[UMLAUF_ID])

print("BEST VEHICLE (anchors) auto :", best_vehicle_auto)
print("BEST VEHICLE (anchors) final:", best_vehicle_final)
print("anchors:", len(df_anchor), "| matched:", int(df_anchor["best_vehicle_id"].notna().sum()))

display(agg_vehicle.head(20))

# ab hier IMMER best_vehicle_final verwenden
# z.B.:
# arrivals = compute_arrivals_anchor_interp(soll_stops, soll_segments, vp, best_vehicle_final)


In [ ]:
# 2026-02-02 18:10 Europe/Berlin

def time_window(soll_fahrt: gpd.GeoDataFrame):
    return (
        soll_fahrt["ts_soll"].min() - TIME_GATE,
        soll_fahrt["ts_soll"].max() + TIME_GATE
    )


def likelihood_match_fahrt_metric(
    soll_segments_fahrt_wgs: gpd.GeoDataFrame,
    vp_wgs: gpd.GeoDataFrame,
    vehicle_id,
    t_min,
    t_max,
    max_dist_m: float = LIKELIHOOD_MAX_DIST_M,
    sigma_m: float = SIGMA_DIST_M,
) -> gpd.GeoDataFrame:
    segs = ensure_gdf(soll_segments_fahrt_wgs, crs="EPSG:4326")
    vp0 = ensure_gdf(vp_wgs, crs="EPSG:4326")

    vp0 = vp0.copy()
    vp0["ts"] = normalize_time_utc(vp0["ts"], assume_local_if_naive=False)

    ist = vp0[(vp0["vehicle_id"] == vehicle_id) & (vp0["ts"] >= t_min) & (vp0["ts"] <= t_max)].copy()
    if ist.empty or segs.empty:
        return gpd.GeoDataFrame(columns=["ts","geometry","s_hat","confidence"], geometry="geometry", crs=vp0.crs)

    segs_m = to_metric(segs)
    metric_crs = segs_m.crs
    ist_m = ist.to_crs(metric_crs)

    sidx = segs_m.sindex
    gc_seg = segs_m.geometry.name
    gc_ist = ist_m.geometry.name

    rows = []

    for idx, r in ist_m.iterrows():
        pt: Point = r[gc_ist]
        if pt is None or pt.is_empty:
            continue

        buf = pt.buffer(max_dist_m)
        cand_idx = list(sidx.intersection(buf.bounds)) if sidx is not None else []
        if not cand_idx:
            continue

        cand = segs_m.iloc[cand_idx].copy()
        cand["dist_m"] = cand[gc_seg].distance(pt)
        cand = cand[cand["dist_m"] <= max_dist_m].copy()
        if cand.empty:
            continue

        s_list, w_list = [], []

        for _, seg in cand.iterrows():
            line: LineString = seg[gc_seg]
            proj_len = float(line.project(pt))
            proj_pt = line.interpolate(proj_len)
            dist_m = float(pt.distance(proj_pt))

            geom_len = float(line.length) if float(line.length) > 0 else 1.0
            frac = float(np.clip(proj_len / geom_len, 0.0, 1.0))
            seg_len = float(seg["cum_end_m"] - seg["cum_start_m"])
            s_abs = float(seg["cum_start_m"] + frac * seg_len)

            w = float(np.exp(-0.5 * (dist_m / sigma_m) ** 2))
            s_list.append(s_abs)
            w_list.append(w)

        if not s_list:
            continue

        s_arr = np.asarray(s_list, dtype=float)
        w_arr = np.asarray(w_list, dtype=float)

        s_hat = float((s_arr * w_arr).sum() / w_arr.sum())
        conf = float(w_arr.sum())

        rows.append({
            "ts": ist.loc[idx, "ts"],  # UTC-aware
            "geometry": ist.loc[idx, ist.geometry.name],
            "s_hat": s_hat,
            "confidence": conf,
            "vp_id": ist.loc[idx, "vp_id"] if "vp_id" in ist.columns else np.nan,
        })

    if not rows:
        return gpd.GeoDataFrame(columns=["ts","geometry","s_hat","confidence"], geometry="geometry", crs=vp0.crs)

    return gpd.GeoDataFrame(rows, geometry="geometry", crs=vp0.crs)


def project_stop_to_route_metric(stop_point_wgs: Point, segs_fahrt_wgs: gpd.GeoDataFrame) -> dict:
    segs = ensure_gdf(segs_fahrt_wgs, crs="EPSG:4326")
    if segs.empty or stop_point_wgs is None or stop_point_wgs.is_empty:
        return {"s_stop": np.nan, "dist_m": np.nan}

    segs_m = to_metric(segs)
    metric_crs = segs_m.crs
    stop_m = gpd.GeoSeries([stop_point_wgs], crs="EPSG:4326").to_crs(metric_crs).iloc[0]
    gc = segs_m.geometry.name

    best = None
    for _, seg in segs_m.iterrows():
        line: LineString = seg[gc]
        proj_len = float(line.project(stop_m))
        proj_pt = line.interpolate(proj_len)
        dist_m = float(stop_m.distance(proj_pt))

        geom_len = float(line.length) if float(line.length) > 0 else 1.0
        frac = float(np.clip(proj_len / geom_len, 0.0, 1.0))
        seg_len = float(seg["cum_end_m"] - seg["cum_start_m"])
        s_abs = float(seg["cum_start_m"] + frac * seg_len)

        if best is None or dist_m < best["dist_m"]:
            best = {"s_stop": s_abs, "dist_m": dist_m}

    return best if best is not None else {"s_stop": np.nan, "dist_m": np.nan}


# 2026-02-03 10:49 Europe/Berlin

from datetime import timedelta
import numpy as np
import pandas as pd
import geopandas as gpd


def detect_anchors_strict(
    soll_fahrt_wgs: gpd.GeoDataFrame,     # Stops der einen Fahrt (für Geometrie)
    soll_stops_proj: pd.DataFrame,        # hat stop_seq, s_stop, ts_soll (UTC-aware)
    traj_wgs: gpd.GeoDataFrame,           # Output aus likelihood_match_fahrt_metric (ts, geometry, s_hat_dir, vp_id)
    dist_m: float,
    max_dt: timedelta,
) -> dict:
    """
    Strikte Anker:
    - Kandidat muss räumlich <= dist_m am Stop sein
    - und zeitlich |ts - ts_soll| <= max_dt
    - Auswahl: minimal |dt|, tie-break: minimal dist

    NEU (Debug):
    anchors[stop_seq] = {
        "ts": ..., "s": ...,
        "vp_id": ..., "ist_lon": ..., "ist_lat": ...
    }
    """
    anchors = {}

    if traj_wgs is None or len(traj_wgs) == 0 or soll_fahrt_wgs is None or len(soll_fahrt_wgs) == 0:
        return anchors

    # metrisch (für echte Meter-Distanz)
    metric_crs = soll_fahrt_wgs.estimate_utm_crs()
    stops_m = soll_fahrt_wgs.to_crs(metric_crs).copy()
    traj_m  = traj_wgs.to_crs(metric_crs).copy()

    gc_stop = stops_m.geometry.name
    gc_traj = traj_m.geometry.name

    sidx = traj_m.sindex

    stop_geom_by_seq = (
        stops_m.set_index("stop_seq")[gc_stop].to_dict()
        if "stop_seq" in stops_m.columns else {}
    )

    # WGS-Geometriespalte (für lon/lat)
    gc_traj_wgs = traj_wgs.geometry.name

    for _, stop in soll_stops_proj.iterrows():
        seq = int(stop.stop_seq)
        ts_soll = stop.ts_soll

        pt = stop_geom_by_seq.get(seq, None)
        if pt is None or pt.is_empty or pd.isna(ts_soll):
            continue

        buf = pt.buffer(dist_m)
        idx = list(sidx.intersection(buf.bounds)) if sidx is not None else []
        if not idx:
            continue

        cand = traj_m.iloc[idx].copy()
        cand["dist_m"] = cand[gc_traj].distance(pt)
        cand = cand[cand["dist_m"] <= dist_m].copy()
        if cand.empty:
            continue

        cand["dt_s"] = (cand["ts"] - ts_soll).abs().dt.total_seconds()
        cand = cand[cand["dt_s"] <= max_dt.total_seconds()].copy()
        if cand.empty:
            continue

        cand = cand.sort_values(["dt_s", "dist_m"], ascending=[True, True])
        best = cand.iloc[0]
        best_idx = best.name  # <- bleibt der Original-Index aus traj_wgs

        # IST-Koordinaten (WGS84) vom gleichen Punkt
        pt_wgs = traj_wgs.loc[best_idx, gc_traj_wgs]
        ist_lon = float(pt_wgs.x) if pt_wgs is not None and (not pt_wgs.is_empty) else np.nan
        ist_lat = float(pt_wgs.y) if pt_wgs is not None and (not pt_wgs.is_empty) else np.nan

        anchors[seq] = {
            "ts": best["ts"],
            "s": float(best["s_hat_dir"]),
            "vp_id": best["vp_id"] if "vp_id" in best.index else np.nan,
            "ist_lon": ist_lon,
            "ist_lat": ist_lat,

            # optional (hilft beim Debuggen)
            # "dt_s": float(best["dt_s"]),
            # "dist_m": float(best["dist_m"]),
        }

    return anchors




def interpolate_between_anchors(stop_seq, s_stop, anchors, soll_stops_proj):
    seqs = soll_stops_proj["stop_seq"].values
    idx = np.where(seqs == stop_seq)[0][0]

    if idx == 0 or idx == len(seqs) - 1:
        return None

    prev_seq, next_seq = seqs[idx - 1], seqs[idx + 1]
    if prev_seq not in anchors or next_seq not in anchors:
        return None

    a0, a1 = anchors[prev_seq], anchors[next_seq]
    denom = (a1["s"] - a0["s"])
    if denom == 0:
        return None

    ratio = (s_stop - a0["s"]) / denom
    ratio = np.clip(ratio, 0, 1)

    return a0["ts"] + ratio * (a1["ts"] - a0["ts"])


def trajectory_interpolation(traj: pd.DataFrame, s_stop: float):
    s = traj["s_hat_dir"].values
    t = traj["ts"].astype("int64").values  # ns (tz-aware -> epoch ns)

    if len(s) == 0:
        return None, None

    s_min, s_max = np.nanmin(s), np.nanmax(s)

    if s_min <= s_stop <= s_max:
        ts_ns = int(np.interp(s_stop, s, t))
        return pd.to_datetime(ts_ns, utc=True), "trajectory_interp"

    if len(s) >= 2:
        i0, i1 = (0, 1) if s_stop < s_min else (-2, -1)
        ds = s[i1] - s[i0]
        dt = (t[i1] - t[i0]) / 1e9
        if ds > 0 and dt > 0:
            v = ds / dt
            if 0.5 < v < 30:
                dt_extra = (s_stop - s[i0]) / v
                if abs(dt_extra) <= 600:
                    ts_ns = int(t[i0] + dt_extra * 1e9)
                    return pd.to_datetime(ts_ns, utc=True), "trajectory_extrapol"

    return None, None


# 2026-02-03 11:16 Europe/Berlin

def compute_arrivals_anchor_interp(
    soll_stops_wgs: gpd.GeoDataFrame,
    soll_segments_wgs: gpd.GeoDataFrame,
    vp_wgs: gpd.GeoDataFrame,
    vehicle_id,
    manual_vehicle_id=None,          # <- NEU: global override
    manual_vehicle_by_frt=None,      # <- NEU: dict pro frt_fid
) -> pd.DataFrame:

    manual_vehicle_by_frt = manual_vehicle_by_frt or {}

    soll_stops0 = ensure_gdf(soll_stops_wgs, crs="EPSG:4326").copy()
    soll_segments0 = ensure_gdf(soll_segments_wgs, crs="EPSG:4326").copy()
    vp0 = ensure_gdf(vp_wgs, crs="EPSG:4326").copy()

    soll_stops0["ts_soll"] = normalize_time_utc(soll_stops0["ts_soll"], assume_local_if_naive=True)
    vp0["ts"] = normalize_time_utc(vp0["ts"], assume_local_if_naive=False)

    soll_stops0 = soll_stops0.sort_values(["frt_fid", "stop_seq"]).reset_index(drop=True)
    geom_col_soll = soll_stops0.geometry.name

    all_rows = []

    for frt_fid, soll_fahrt in soll_stops0.groupby("frt_fid"):
        segs_fahrt = soll_segments0[soll_segments0["frt_fid"] == frt_fid].copy()
        if segs_fahrt.empty:
            continue

        # --- Vehicle-ID 결정: pro Fahrt -> global override -> default ---
        vid = vehicle_id
        if frt_fid in manual_vehicle_by_frt and manual_vehicle_by_frt[frt_fid] is not None:
            vid = manual_vehicle_by_frt[frt_fid]
        if manual_vehicle_id is not None:
            vid = manual_vehicle_id

        # cast auf dtype von vp0['vehicle_id']
        vid = _cast_vehicle_id_like(vp0, vid)

        # Stops -> s_stop
        proj_rows = []
        for _, row in soll_fahrt.sort_values("stop_seq").iterrows():
            pt = row[geom_col_soll]
            res = project_stop_to_route_metric(pt, segs_fahrt)
            proj_rows.append({
                "stop_seq": int(row["stop_seq"]),
                "s_stop": float(res["s_stop"]),
                "ts_soll": row["ts_soll"],
                "fahrzeit_sek": row.get("fahrzeit_sek", np.nan),
            })
        soll_stops_proj = pd.DataFrame(proj_rows).sort_values("stop_seq").reset_index(drop=True)

        # IST -> Trajektorie (Likelihood) mit *dem gewählten* Vehicle
        t_min, t_max = time_window(soll_fahrt)
        ist_on_route = likelihood_match_fahrt_metric(
            segs_fahrt, vp0, vid, t_min, t_max,
            max_dist_m=LIKELIHOOD_MAX_DIST_M,
            sigma_m=SIGMA_DIST_M,
        )
        if ist_on_route.empty:
            continue

        traj = ist_on_route.sort_values("s_hat").reset_index(drop=True)
        traj["s_hat_dir"] = traj["s_hat"].cummax()

        anchors = detect_anchors_strict(
            soll_fahrt_wgs=soll_fahrt,
            soll_stops_proj=soll_stops_proj,
            traj_wgs=traj,
            dist_m=STOP_ANCHOR_DIST_M,
            max_dt=STOP_ANCHOR_MAX_DT,
        )

        for _, stop in soll_stops_proj.iterrows():
            stop_seq = int(stop["stop_seq"])
            s_stop = float(stop["s_stop"])

            if stop_seq in anchors:
                ts_hat, src = anchors[stop_seq]["ts"], "anchor"
                anchor_ist_lon = anchors[stop_seq].get("ist_lon", np.nan)
                anchor_ist_lat = anchors[stop_seq].get("ist_lat", np.nan)
                anchor_vp_id   = anchors[stop_seq].get("vp_id", pd.NA)
            else:
                ts_hat = interpolate_between_anchors(stop_seq, s_stop, anchors, soll_stops_proj)
                src = "anchor_interp" if ts_hat is not None else None
                anchor_ist_lon, anchor_ist_lat, anchor_vp_id = np.nan, np.nan, pd.NA

            if ts_hat is None:
                ts_hat, src2 = trajectory_interpolation(traj, s_stop)
                src = src2
                anchor_ist_lon, anchor_ist_lat, anchor_vp_id = np.nan, np.nan, pd.NA

            all_rows.append({
                "frt_fid": frt_fid,
                "stop_seq": stop_seq,
                "ankunft_ist": ts_hat,
                "arrival_source": src,
                "vehicle_id": vid,  # <- wichtig: das tatsächlich genutzte vehicle_id

                "anchor_ist_lon": anchor_ist_lon,
                "anchor_ist_lat": anchor_ist_lat,
                "anchor_vp_id": anchor_vp_id,
            })

    return pd.DataFrame(all_rows)



def fill_edge_gaps_two_stage(
    final_stops_wgs: gpd.GeoDataFrame,
    vp_wgs: gpd.GeoDataFrame,
    vehicle_id,
    stages=None,
    only_first_last=True,
):
    if stages is None:
        stages = [
            ("anchor_edge_narrow", EDGE_STAGE_1_TIME, EDGE_STAGE_1_DIST),
            ("anchor_edge_wide",   EDGE_STAGE_2_TIME, EDGE_STAGE_2_DIST),
        ]

    final_stops = ensure_gdf(final_stops_wgs, crs="EPSG:4326").copy()
    vp0 = ensure_gdf(vp_wgs, crs="EPSG:4326").copy()

    final_stops["ankunft_soll"] = normalize_time_utc(final_stops["ankunft_soll"], assume_local_if_naive=True)
    final_stops["ankunft_ist"]  = normalize_time_utc(final_stops["ankunft_ist"],  assume_local_if_naive=False)
    vp0["ts"] = normalize_time_utc(vp0["ts"], assume_local_if_naive=False)

    vp0 = vp0[vp0["vehicle_id"] == vehicle_id].copy()
    if vp0.empty:
        print("⚠️ Keine VP für vehicle_id gefunden:", vehicle_id)
        return final_stops

    metric_crs = final_stops.estimate_utm_crs()
    stops_m = final_stops.to_crs(metric_crs)
    vp_m = vp0.to_crs(metric_crs)

    sidx = vp_m.sindex
    gc_stop = stops_m.geometry.name
    gc_vp = vp_m.geometry.name

    if only_first_last:
        edge_idx = []
        for frt, grp in final_stops.groupby("frt_fid"):
            g = grp.sort_values("stop_seq")
            if len(g) == 0:
                continue
            edge_idx.append(g.index[0])
            edge_idx.append(g.index[-1])
        edge_mask = final_stops.index.isin(edge_idx)
    else:
        edge_mask = np.ones(len(final_stops), dtype=bool)

    gap_mask = final_stops["ankunft_ist"].isna() & edge_mask
    gaps = final_stops.loc[gap_mask].copy()
    if gaps.empty:
        print("✅ Keine Edge-Gaps zum Füllen.")
        return final_stops

    filled_total = 0

    for idx, row in gaps.iterrows():
        ts_soll = row["ankunft_soll"]
        if pd.isna(ts_soll):
            continue

        pt = stops_m.loc[idx, gc_stop]
        if pt is None or pt.is_empty:
            continue

        for stage_name, time_buf, dist_m in stages:
            t0 = ts_soll - time_buf
            t1 = ts_soll + time_buf

            buf = pt.buffer(dist_m)
            cand_idx = list(sidx.intersection(buf.bounds)) if sidx is not None else []
            if not cand_idx:
                continue

            cand = vp_m.iloc[cand_idx].copy()
            cand["dist_m"] = cand[gc_vp].distance(pt)
            cand = cand[cand["dist_m"] <= dist_m].copy()
            if cand.empty:
                continue

            cand = cand[(cand["ts"] >= t0) & (cand["ts"] <= t1)].copy()
            if cand.empty:
                continue

            cand["dt_s"] = (cand["ts"] - ts_soll).abs().dt.total_seconds()
            cand = cand.sort_values(["dt_s", "dist_m"])
            best = cand.iloc[0]

            final_stops.loc[idx, "ankunft_ist"] = best["ts"]
            prev_src = final_stops.loc[idx, "arrival_source"]
            final_stops.loc[idx, "arrival_source"] = (
                stage_name if pd.isna(prev_src) else f"{prev_src}|{stage_name}"
            )
            filled_total += 1
            break

    print(f"✅ Edge-Gaps gefüllt: {filled_total}/{len(gaps)} (2-stufig)")
    return final_stops


In [ ]:
# 2026-02-02 18:10 Europe/Berlin

if best_vehicle_final is None:
    raise ValueError("best_vehicle_final ist None – Anchor-Matching hat kein Fahrzeug gefunden.")


arrivals = compute_arrivals_anchor_interp(
    soll_stops_wgs=soll_stops,
    soll_segments_wgs=soll_segments,
    vp_wgs=vp,
    vehicle_id=best_vehicle_final,
)



print("Arrivals rows:", len(arrivals))
if len(arrivals):
    print(arrivals["arrival_source"].value_counts(dropna=False))

# Merge auf Stops
final_stops = (
    soll_stops.merge(arrivals, on=["frt_fid", "stop_seq"], how="left")
    .rename(columns={"ts_soll": "ankunft_soll"})
)

# Edge-Gaps füllen (zweistufig, nur first/last je Fahrt)
final_stops = fill_edge_gaps_two_stage(
    final_stops_wgs=final_stops,
    vp_wgs=vp,
    vehicle_id=best_vehicle_final,
    stages=[
        ("anchor_edge_narrow", EDGE_STAGE_1_TIME, EDGE_STAGE_1_DIST),
        ("anchor_edge_wide",   EDGE_STAGE_2_TIME, EDGE_STAGE_2_DIST),
    ],
    only_first_last=True
)

missing = final_stops["ankunft_ist"].isna().mean()
print(f"Missing ankunft_ist: {missing:.1%}")

# -----------------------------
# CSV export (Geometry raus)
# -----------------------------
csv_out = EXPORT_DIR / f"stops_arrivals_anchor_interp_{UMLAUF_ID}_{TAG}.csv"
geom_drop = [c for c in ["geometry", "geom"] if c in final_stops.columns]
final_stops.drop(columns=geom_drop, errors="ignore").to_csv(csv_out, index=False)
print("✅ CSV:", csv_out)

# -----------------------------
# HTML Map export (wie bei dir)
# -----------------------------
stops_wgs = final_stops.to_crs("EPSG:4326").copy()
segs_wgs = soll_segments.to_crs("EPSG:4326").copy()
vp_best = vp[vp["vehicle_id"] == best_vehicle_final].to_crs("EPSG:4326").copy()

gc = stops_wgs.geometry.name
center_lat = float(stops_wgs[gc].y.mean())
center_lon = float(stops_wgs[gc].x.mean())

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, control_scale=True)
Fullscreen(position="topright").add_to(m)
MousePosition().add_to(m)
MeasureControl(position="topright", primary_length_unit="meters").add_to(m)

# SOLL Segmente (grau)
for _, r in segs_wgs.sort_values(["frt_fid", "edge_idx"]).iterrows():
    ls = r[segs_wgs.geometry.name]
    if ls is None or ls.is_empty:
        continue
    coords = [(lat, lon) for lon, lat in list(ls.coords)]
    folium.PolyLine(coords, weight=3, opacity=0.35, color="#777777").add_to(m)

fg_anchor = folium.FeatureGroup(name="Stops: anchor", show=True)
fg_interp = folium.FeatureGroup(name="Stops: anchor_interp", show=True)
fg_traj   = folium.FeatureGroup(name="Stops: traj fallback", show=False)
fg_edge   = folium.FeatureGroup(name="Stops: edge fill", show=True)
fg_none   = folium.FeatureGroup(name="Stops: no arrival", show=False)

def _src_has(src, key):
    return (isinstance(src, str) and (key in src))

for _, r in stops_wgs.sort_values(["frt_fid", "stop_seq"]).iterrows():
    pt = r[gc]
    if pt is None or pt.is_empty:
        continue

    src = r.get("arrival_source", None)
    tip = f"frt={r['frt_fid']} seq={r['stop_seq']} | soll={r.get('ankunft_soll')} | ist={r.get('ankunft_ist')} | src={src}"

    if src == "anchor":
        color = "#2ca02c"; layer = fg_anchor
    elif src == "anchor_interp":
        color = "#ff7f0e"; layer = fg_interp
    elif src in ("trajectory_interp", "trajectory_extrapol"):
        color = "#1f77b4"; layer = fg_traj
    elif _src_has(src, "anchor_edge_"):
        color = "#9467bd"; layer = fg_edge
    else:
        color = "#d62728"; layer = fg_none

    folium.CircleMarker(
        [float(pt.y), float(pt.x)],
        radius=4,
        color=color,
        fill=True,
        fill_opacity=0.9,
        tooltip=tip
    ).add_to(layer)

fg_anchor.add_to(m)
fg_interp.add_to(m)
fg_traj.add_to(m)
fg_edge.add_to(m)
fg_none.add_to(m)

# IST Punkte (sample)
vp_best = vp_best.copy()
vp_best["ts"] = normalize_time_utc(vp_best["ts"], assume_local_if_naive=False)

fg_ist = folium.FeatureGroup(name=f"IST VP (vehicle={best_vehicle_final}) sample", show=False)
MAX_PLOT = 40000
step = max(1, int(len(vp_best) / MAX_PLOT)) if len(vp_best) > MAX_PLOT else 1

gcv = vp_best.geometry.name
for _, r in vp_best.sort_values("ts").iloc[::step].iterrows():
    pt = r[gcv]
    if pt is None or pt.is_empty:
        continue
    folium.CircleMarker(
        [float(pt.y), float(pt.x)],
        radius=2,
        color="#9467bd",
        fill=True,
        fill_opacity=0.6,
        tooltip=f"ts={r.get('ts')}"
    ).add_to(fg_ist)

fg_ist.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

html_out = EXPORT_DIR / f"stops_arrivals_anchor_interp_{UMLAUF_ID}_{TAG}.html"
m.save(str(html_out))
print("✅ HTML:", html_out)

webbrowser.open(f"file://{html_out.resolve()}")
